# RSNA Knee Abnormality Detection — Report Baseline

Phase 3A's first leakage-safe competition baseline: a frozen character n-gram TF-IDF vectorizer plus one-vs-rest logistic regression, trained only on the 58 human-labeled studies' report text. Produces deterministic out-of-fold evidence, refits on all 58 labeled studies, and predicts the held-out test set. This notebook processes real report text directly, but only ever displays aggregate counts, rates, and summary statistics — no report excerpts, no per-study prediction tables, no study-identifier lists, no row-level probabilities.

Every result below is computed live when this notebook executes on Kaggle — none is asserted here in advance.

In [ ]:
import hashlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

# Locate the one attached rsna-knee-mri-src dataset root -- it holds
# both the knee_mri source package and the vendored offline wheel --
# and verify/install the pinned wheel before inserting the source
# path or importing knee_mri anywhere in this notebook.
package_initializers = tuple(
    Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py")
)
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
_src_root = package_initializers[0].parent.parent
_dataset_root = _src_root.parent

WHEEL_NAME = "iterative_stratification-0.1.9-py3-none-any.whl"
EXPECTED_SHA256 = "476f8deff6753fb1725612fe41e59cc2058f8f2524ae5d1ccee88eb8c8d3de80"

wheel_matches = tuple(_dataset_root.rglob(WHEEL_NAME))
if len(wheel_matches) != 1:
    raise RuntimeError("Expected exactly one pinned iterative-stratification wheel.")
wheel_path = wheel_matches[0]
if hashlib.sha256(wheel_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pinned iterative-stratification wheel checksum mismatch.")

try:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", str(wheel_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the pinned wheel.") from None
if install_result.returncode != 0:
    raise RuntimeError("Offline installation of the pinned wheel failed.")
if importlib.metadata.version("iterative-stratification") != "0.1.9":
    raise RuntimeError("Installed iterative-stratification version mismatch.")

sys.path.insert(0, str(_src_root))

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

## 1. Frozen Experiment Contract

In [ ]:
from knee_mri.report_model import build_report_classifier, build_report_vectorizer

_vectorizer = build_report_vectorizer()
_classifier = build_report_classifier()

frozen_contract = pd.DataFrame(
    [
        {"Setting": "TF-IDF analyzer", "Value": _vectorizer.analyzer},
        {"Setting": "TF-IDF n-gram range", "Value": str(_vectorizer.ngram_range)},
        {"Setting": "TF-IDF min_df", "Value": _vectorizer.min_df},
        {"Setting": "TF-IDF max_features", "Value": _vectorizer.max_features},
        {"Setting": "TF-IDF sublinear_tf", "Value": _vectorizer.sublinear_tf},
        {"Setting": "TF-IDF lowercase", "Value": _vectorizer.lowercase},
        {"Setting": "TF-IDF strip_accents", "Value": _vectorizer.strip_accents},
        {"Setting": "Classifier penalty", "Value": _classifier.estimator.penalty},
        {"Setting": "Classifier solver", "Value": _classifier.estimator.solver},
        {"Setting": "Classifier C", "Value": _classifier.estimator.C},
        {"Setting": "Classifier class_weight", "Value": _classifier.estimator.class_weight},
        {"Setting": "Classifier max_iter", "Value": _classifier.estimator.max_iter},
        {"Setting": "Classifier random_state", "Value": _classifier.estimator.random_state},
        {"Setting": "One-vs-rest n_jobs", "Value": _classifier.n_jobs},
        {"Setting": "Fold candidates", "Value": "(5, 4, 3, 2), first feasible, no retry"},
        {"Setting": "Fold seed", "Value": SEED},
    ]
).set_index("Setting")

display(frozen_contract)

**Interpretation.** These settings are frozen before any result is viewed — no hyperparameter search, seed retry, or fold-count adjustment happens after this point. The attached, independently tested package provides this implementation; this notebook only orchestrates it.

## 2. Offline Setup and Data Validation

In [ ]:
from knee_mri.dataset import prepare_modeling_inputs
from knee_mri.labels import LABEL_COLUMNS

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

inputs = prepare_modeling_inputs(train_df, test_df, sample_df)
y = inputs.labeled_studies[LABEL_COLUMNS]

data_summary = pd.Series(
    {
        "Labeled studies": len(inputs.labeled_studies),
        "Test studies": len(inputs.test_studies),
        "Missing/blank test reports": inputs.missing_test_report_count,
    },
    name="Count",
).to_frame()

display(data_summary)

**Interpretation.** The offline wheel was verified by exact filename and SHA-256 and installed before any `knee_mri` import in this notebook — no runtime download, no fallback version. `prepare_modeling_inputs` validates schemas, identifiers, and report types before any modeling begins; a validation failure stops the notebook here rather than letting a malformed input reach the model.

## 3. Deterministic Multilabel Folds

In [ ]:
from knee_mri.model_selection import select_multilabel_folds

selected_fold_count, folds = select_multilabel_folds(y, seed=SEED)

selected_fold_summary = pd.Series(
    {"Selected fold count": selected_fold_count}, name="Value"
).to_frame()

fold_sizes = pd.DataFrame(
    [
        {
            "Fold": fold_index,
            "Training Rows": len(training_indices),
            "Validation Rows": len(validation_indices),
        }
        for fold_index, (training_indices, validation_indices) in enumerate(folds)
    ]
).set_index("Fold")

fold_validation_positive_counts = pd.DataFrame(
    {
        fold_index: y.iloc[validation_indices].sum()
        for fold_index, (_, validation_indices) in enumerate(folds)
    }
)
fold_validation_positive_counts.index.name = "Label"
fold_validation_positive_counts.columns.name = "Fold"

display(selected_fold_summary)
display(fold_sizes)
display(fold_validation_positive_counts)

**Interpretation.** The selected fold count is the first candidate from `(5, 4, 3, 2)` whose every validation fold contains both classes for every label — no alternate seed is tried and no score is consulted during selection. A lower selected count than 5 would indicate the 58-study sample couldn't support finer folds while keeping every label represented in every validation split; the per-label positive counts above confirm each fold's class balance directly.

## 4. Constant-Probability Sanity Check

In [ ]:
from knee_mri.metrics import macro_auc

constant_predictions = pd.DataFrame(0.5, index=y.index, columns=LABEL_COLUMNS)
assert macro_auc(y, constant_predictions) == 0.5

sanity_check = pd.Series(
    {"Constant-0.5 macro AUC": macro_auc(y, constant_predictions)}, name="Value"
).to_frame()
display(sanity_check)

**Interpretation.** A constant 0.5 prediction for every label must score exactly 0.5 macro AUC — this is asserted, not just displayed, so the notebook stops immediately if the metric implementation or label wiring is ever wrong. A pooled or per-label score below this value in Section 6 is a real possible outcome for an anti-predictive or noisy model, not proof of a scoring error — this section only confirms the metric and label wiring are correct for a known input.

## 5. Fold-Local Out-of-Fold Evaluation

In [ ]:
from knee_mri.report_model import cross_validate_report_model

cv_result = cross_validate_report_model(inputs.labeled_studies["Report"], y, folds)

fold_diagnostics = pd.DataFrame(
    {
        "Fold Macro AUC": cv_result.fold_macro_auc,
        "Vocabulary Size": cv_result.vocabulary_sizes,
    }
)
fold_diagnostics.index.name = "Fold"

display(fold_diagnostics)

**Interpretation.** Each fold fits a fresh vectorizer and classifier on its own training rows only, so fold-local vocabulary size reflects that fold's training text alone. Fold macro AUC here is one score per selected fold (2-5 values) — a diagnostic, not the primary score. Section 6 instead pools every out-of-fold prediction into one score before averaging across the 12 labels, which is more stable at this sample size than averaging this section's small-sample fold-level scores.

## 6. Pooled and Per-Label Interpretation

In [ ]:
pooled_summary = pd.Series(
    {"Pooled macro AUC": cv_result.pooled_macro_auc}, name="Value"
).to_frame()
per_label_summary = (
    pd.Series(cv_result.pooled_per_label_auc, name="Pooled AUC")
    .rename_axis("Label")
    .to_frame()
)

display(pooled_summary)
display(per_label_summary)

**Interpretation.** Pooled macro AUC is the primary internal score for this baseline: every study's out-of-fold prediction is pooled per label before scoring, and the 12 per-label scores above are then averaged — distinct from Section 5's 2-5 fold-level diagnostic scores. This is internal cross-validation on the only 58 labeled studies, not an independent confirmation set — no configuration was chosen by looking at this result.

## 7. Full-Data Refit and Test Prediction

In [ ]:
from knee_mri.report_model import fit_report_model

vectorizer, classifier = fit_report_model(inputs.labeled_studies["Report"], y)
test_features = vectorizer.transform(inputs.test_studies["Report"])
test_probabilities = classifier.predict_proba(test_features)

test_probability_summary = pd.DataFrame(
    test_probabilities, columns=LABEL_COLUMNS
).describe().T[["mean", "std", "min", "max"]]

display(test_probability_summary)

**Interpretation.** The refit vectorizer and classifier are fit fresh on all 58 labeled studies with the same frozen settings as every cross-validation fold — no configuration changes after seeing Section 6's result. Only aggregate per-label probability statistics are shown; no individual test study's prediction is displayed.

## 8. Submission Validation and Artifact

In [ ]:
from knee_mri.submission import build_submission

submission = build_submission(
    sample_df,
    inputs.test_studies["StudyInstanceUID"],
    test_probabilities,
)
submission.to_csv("/kaggle/working/submission.csv", index=False)

submission_summary = pd.Series(
    {"Submission rows": len(submission), "Submission columns": submission.shape[1]},
    name="Value",
).to_frame()
display(submission_summary)

**Interpretation.** `build_submission` validates schema, identifier order, and probability range before anything is written; a validation failure stops the notebook rather than writing a malformed `submission.csv`. Only the artifact's shape is shown here — no per-study probability row is displayed.

## 9. Limitations and Phase 3B

- This is internal cross-validation on the 58 human-labeled studies, not an independent confirmation set — no configuration was chosen by looking at any score in this notebook.
- Only report text is used; MRI-derived features are out of scope for this phase and are a candidate for a future imaging baseline.
- The frozen character n-gram vectorizer and one-vs-rest logistic regression are the smallest, most interpretable first baseline — a future imaging branch would be evaluated only after this baseline's own results exist.